# Qualidade de Vinhos — Projeto N3 de Análise Preditiva e MLOps
### Notebook completo | UCI Wine Quality • Regressão • Ridge × Random Forest

**Objetivo:** prever a nota numérica `quality` com as 11 medidas físico-químicas de vinhos tintos, comparar os erros de dois regressores e demonstrar o fluxo MLOps em **dois ciclos**.

**Requisitos cobertos:** ingestão e limpeza (ETL); partição treino/validação/teste (60/20/20); treino progressivo com dois lotes; seleção independente por `SelectKBest`, Random Forest e `RFE`; aprovação de atributos por **votação 2/3**; Ridge e Random Forest Regressor; **MAE** como métrica principal, RMSE complementar; **MLflow**; promoção controlada do modelo; teste isolado ao final; previsão; gráficos de erros e atributos.

A execução cria `data/`, `reports/`, `models/`, `mlruns/` e `evidencias/` no diretório de trabalho. Se o arquivo estiver dentro de uma pasta chamada `notebooks`, usa a pasta-mãe como raiz.

> **Metodologia:** a base UCI é estática. Os dois ciclos *simulam* a chegada de lotes progressivos, não um streaming real. Nenhum resultado de MAE será estimado sem executar o treinamento.

## 0. Preparação do ambiente

- Recomendado: **Python 3.11**.
- Instale as dependências do projeto pelo terminal: `pip install -r requirements.txt` ou, com o notebook isolado, execute `%pip install numpy pandas scikit-learn joblib mlflow matplotlib` em uma nova célula.
- Para os arquivos complementares, use o pacote ZIP do projeto, que também inclui Streamlit, pytest e GitHub Actions.
- O download inicial da UCI exige internet. Se necessário, coloque `winequality-red.csv` em `data/raw/` antes de executar.

**Referência da base:** [UCI Machine Learning Repository — Wine Quality](https://archive.ics.uci.edu/dataset/186/wine+quality).

In [ ]:
from __future__ import annotations

import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import urlretrieve

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE, SelectKBest, f_regression
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    import mlflow
    import mlflow.sklearn
except ImportError as error:
    raise ImportError('MLflow não instalado. Execute: %pip install mlflow==2.19.0') from error

ROOT = Path.cwd().resolve()
if ROOT.name.lower() == 'notebooks':
    ROOT = ROOT.parent
CONFIG = ROOT / 'config' / 'params.json'
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
REPORTS = ROOT / 'reports'
MODELS = ROOT / 'models'
MLRUNS = ROOT / 'mlruns'
EVIDENCIAS = ROOT / 'evidencias'

CONFIG.parent.mkdir(parents=True, exist_ok=True)
if not CONFIG.exists():
    CONFIG.write_text(json.dumps({
        'dataset_url': 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv',
        'random_state': 42, 'test_size': 0.20, 'validation_size_from_dev': 0.25,
        'k_features_per_method': 6, 'minimum_votes': 2,
        'min_mae_improvement': 0.0, 'rf_n_estimators': 200, 'ridge_alpha': 1.0,
    }, indent=2), encoding='utf-8')

print('Raiz do experimento:', ROOT)
print('Python / dependências importadas. Configuração:', CONFIG)

In [ ]:
def ensure_dirs() -> None:
    for folder in (RAW, PROCESSED, REPORTS, MODELS, MLRUNS, EVIDENCIAS):
        folder.mkdir(parents=True, exist_ok=True)


def get_config() -> dict:
    return json.loads(CONFIG.read_text(encoding="utf-8"))


def write_json(path: Path, obj: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

In [ ]:
ensure_dirs()
cfg = get_config()
print('Parâmetros reprodutíveis:')
display(pd.DataFrame(list(cfg.items()), columns=['Parâmetro', 'Valor']))

## 1. ETL — aquisição, limpeza, validação de esquema e partição

A base original de **vinhos tintos** possui 11 atributos numéricos e o alvo `quality`. O CSV oficial utiliza o separador `;`. Na limpeza, convertemos as colunas a números, excluímos linhas com valores ausentes e duplicatas e restringimos a nota ao intervalo indicado. Geramos um manifesto com as contagens e o hash SHA-256 do arquivo bruto.

O **teste (20%)** e a **validação (20%)** são separados antes dos ciclos. O restante, **treino (60%)**, é dividido em dois lotes. Os conjuntos de validação e teste permanecem fixos nos dois ciclos.

In [ ]:
TARGET = "quality"

FEATURES = [
    "fixed acidity", "volatile acidity", "citric acid", "residual sugar",
    "chlorides", "free sulfur dioxide", "total sulfur dioxide", "density",
    "pH", "sulphates", "alcohol",
]


def prepare_data() -> dict:
    """Separa treino em 2 lotes e fixa validação/teste para ambos os ciclos."""
    ensure_dirs()
    config = get_config()
    source = RAW / "winequality-red.csv"
    if not source.exists():
        try:
            urlretrieve(config["dataset_url"], source)
        except Exception as error:
            source.unlink(missing_ok=True)
            raise RuntimeError(
                "Falha ao baixar a base UCI. Baixe 'winequality-red.csv' da UCI "
                "e salve em data/raw/winequality-red.csv antes de executar novamente."
            ) from error

    # A origem usa ';'. Não substituir silenciosamente outra base.
    df = pd.read_csv(source, sep=";")
    expected = FEATURES + [TARGET]
    if sorted(df.columns) != sorted(expected):
        raise ValueError(
            f"Colunas inesperadas. Esperado: {expected}. Encontrado: {list(df.columns)}"
        )
    total_original = len(df)
    df = df[expected].apply(pd.to_numeric, errors="coerce")
    missing_count = int(df.isna().any(axis=1).sum())
    df = df.dropna()
    repeated_count = int(df.duplicated().sum())
    df = df.drop_duplicates()
    invalid_target = int((~df[TARGET].between(0, 10)).sum())
    df = df.loc[df[TARGET].between(0, 10)].reset_index(drop=True)
    if len(df) < 50:
        raise ValueError("Dados insuficientes para treino, validação e teste.")

    # Divisão total: treino 60%, validação 20%, teste 20%.
    dev, test = train_test_split(
        df, test_size=config["test_size"], random_state=config["random_state"]
    )
    train, validation = train_test_split(
        dev,
        test_size=config["validation_size_from_dev"],
        random_state=config["random_state"],
    )
    train = train.sample(frac=1, random_state=config["random_state"])
    half = len(train) // 2
    parts = {
        "batch_01": train.iloc[:half],
        "batch_02": train.iloc[half:],
        "validation": validation,
        "test": test,
    }
    # Checagem por índice antes de remover o índice na gravação.
    idx_sets = [set(part.index) for part in parts.values()]
    assert len(set.union(*idx_sets)) == len(df)
    assert sum(map(len, idx_sets)) == len(df)

    for name, rows in parts.items():
        rows.to_csv(PROCESSED / f"{name}.csv", index=False)

    manifest = {
        "dataset": "UCI Wine Quality — red wine",
        "source_url": config["dataset_url"],
        "source_sha256": hashlib.sha256(source.read_bytes()).hexdigest(),
        "original_rows": total_original,
        "removed_rows_with_missing": missing_count,
        "removed_duplicate_rows": repeated_count,
        "removed_invalid_quality": invalid_target,
        "clean_rows": len(df),
        "feature_columns": FEATURES,
        "target": TARGET,
        "parts": {name: len(part) for name, part in parts.items()},
        "seed": config["random_state"],
        "protocol": "lote1 -> lote1+lote2; validacao fixa; teste isolado ate ciclo2",
    }
    write_json(REPORTS / "dataset_manifest.json", manifest)
    return manifest

In [ ]:
manifest = prepare_data()
print('Manifesto do dataset gerado:')
display(pd.DataFrame([{'Conjunto': x, 'Linhas': y} for x, y in manifest['parts'].items()]))
print('Linhas originais:', manifest['original_rows'], '| após limpeza:', manifest['clean_rows'])
print('SHA-256 bruto:', manifest['source_sha256'])
print('Relatório:', REPORTS / 'dataset_manifest.json')

# Conferência de partições persistidas.
sets = {name: pd.read_csv(PROCESSED / f'{name}.csv') for name in manifest['parts']}
assert sum(map(len, sets.values())) == manifest['clean_rows']
assert all(list(frame.columns) == FEATURES + [TARGET] for frame in sets.values())

## 2. Exploração inicial de dados (EDA)

Analisamos distribuição de notas e correlação **somente nos lotes de treino**, sem utilizar validação ou teste para orientar a seleção de atributos. As visualizações são descritivas e não substituem os seletores automáticos do passo seguinte.

In [ ]:
treino_exploratorio = pd.concat([sets['batch_01'], sets['batch_02']], ignore_index=True)
print('Resumo numérico das amostras de treinamento:')
display(treino_exploratorio.describe().T.round(3))
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
treino_exploratorio['quality'].value_counts().sort_index().plot.bar(ax=axes[0], color='#84344b')
axes[0].set(title='Notas de qualidade no treino', xlabel='Nota (quality)', ylabel='Quantidade')
corr = treino_exploratorio.corr(numeric_only=True)['quality'].drop('quality').sort_values()
axes[1].barh(corr.index, corr.values, color='#89566a')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set(title='Correlação linear com quality (treino)', xlabel='Coeficiente de Pearson')
plt.show()

## 3. Seleção de atributos por três métodos — votação 2/3

Cada algoritmo indica os **6 atributos mais relevantes**, usando apenas as amostras disponíveis no ciclo:

1. **SelectKBest** com `f_regression`;
2. **Random Forest** por importância das variáveis;
3. **RFE** com `Ridge` aplicado após `StandardScaler` ajustado apenas no treino.

Uma variável só entra nos modelos se receber **pelo menos 2 dos 3 votos**. O processo é recalculado a cada lote; não há atributos pré-aprovados.

In [ ]:
def vote_features(X: pd.DataFrame, y: pd.Series) -> tuple[list[str], pd.DataFrame]:
    cfg = get_config()
    cols = list(X.columns)
    k = min(cfg["k_features_per_method"], len(cols))
    if not cols or len(X) != len(y):
        raise ValueError("Treino inválido ou sem atributos.")

    kbest = SelectKBest(score_func=f_regression, k=k).fit(X, y)
    selected_kbest = set(X.columns[kbest.get_support()])

    forest = RandomForestRegressor(
        n_estimators=cfg["rf_n_estimators"],
        random_state=cfg["random_state"], n_jobs=-1,
    ).fit(X, y)
    importances = pd.Series(forest.feature_importances_, index=cols)
    selected_forest = set(importances.sort_values(ascending=False).head(k).index)

    train_scaled = StandardScaler().fit_transform(X)
    rfe = RFE(Ridge(alpha=cfg["ridge_alpha"]), n_features_to_select=k).fit(train_scaled, y)
    selected_rfe = set(X.columns[rfe.get_support()])

    rows = []
    for col in cols:
        a, b, c = (int(col in selected_kbest), int(col in selected_forest), int(col in selected_rfe))
        rows.append({"feature": col, "SelectKBest": a, "RandomForest": b,
                     "RFE": c, "votes": a + b + c, "selected": (a + b + c) >= cfg["minimum_votes"]})
    votes = pd.DataFrame(rows).sort_values(
        ["votes", "feature"], ascending=[False, True]
    ).reset_index(drop=True)
    chosen = votes.loc[votes["selected"], "feature"].tolist()
    if not chosen:
        raise ValueError("Nenhum atributo aprovado por pelo menos 2 métodos.")
    return chosen, votes

In [ ]:
# Prévia do critério usando apenas o lote 1 (o ciclo 1 fará sua própria seleção).
lote_1 = sets['batch_01']
atributos_preview, votos_preview = vote_features(lote_1[FEATURES], lote_1[TARGET])
print('Atributos aprovados com base apenas no lote 1:', atributos_preview)
display(votos_preview)
assert all(votos_preview.loc[votos_preview['selected'], 'votes'] >= cfg['minimum_votes'])

## 4. Modelagem — Ridge e Random Forest

O **Ridge** padroniza as variáveis dentro de um `Pipeline`, evitando vazamento no `StandardScaler`. O **Random Forest Regressor** modela relações não lineares. Ambos recebem a **mesma lista votada de atributos** no respectivo ciclo. O erro principal é **MAE** (menor é melhor), com RMSE complementar.

In [ ]:
def build_models() -> dict:
    cfg = get_config()
    return {
        "Ridge": Pipeline([
            ("standard_scaler", StandardScaler()),
            ("ridge", Ridge(alpha=cfg["ridge_alpha"])),
        ]),
        "RandomForest": RandomForestRegressor(
            n_estimators=cfg["rf_n_estimators"],
            random_state=cfg["random_state"], n_jobs=-1,
        ),
    }


def evaluate(estimator, X, y) -> dict:
    pred = estimator.predict(X)
    return {
        "mae": float(mean_absolute_error(y, pred)),
        "rmse": float(mean_squared_error(y, pred) ** 0.5),
    }

In [ ]:
print('Modelos configurados:')
for nome, estimador in build_models().items():
    print(f'- {nome}: {estimador}')

## 5. Ciclos progressivos e governança MLOps

- **Ciclo 1:** ajusta três seletores no lote 1, treina Ridge e Random Forest e registra **dois runs no MLflow**. O menor MAE de validação é promovido, pois ainda não existe vigente.
- **Ciclo 2:** repete tudo com lote 1 + lote 2, adiciona **dois novos runs**, compara o vencedor com o modelo vigente no **mesmo conjunto de validação** e **promove apenas se houver melhora estrita**.
- **Encerramento:** avalia uma vez no teste isolado, depois de definida a versão vigente.

O `mlruns/` guarda os experimentos, enquanto `models/current.*` preserva o modelo da inferência. A execução é idempotente: ciclos já concluídos não são reexecutados automaticamente.

In [ ]:
def load_partition(name: str) -> pd.DataFrame:
    return pd.read_csv(PROCESSED / f"{name}.csv")


def run_cycle(requested: str = "auto") -> dict:
    ensure_dirs()
    state_path = REPORTS / "state.json"
    state = json.loads(state_path.read_text(encoding="utf-8")) if state_path.exists() else {"last_cycle": 0}
    last = int(state["last_cycle"])
    cycle = last + 1 if requested == "auto" else int(requested)
    if cycle <= last:
        return {"status": "already_completed", "cycle": cycle}
    if cycle > 2:
        return {"status": "no_new_batch", "cycle": cycle}
    if cycle != last + 1:
        raise ValueError(f"Ciclo inválido. Próximo ciclo permitido: {last + 1}")

    config = get_config()
    manifest_path = REPORTS / "dataset_manifest.json"
    old_manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else None
    manifest = prepare_data()
    if last > 0 and old_manifest and manifest["source_sha256"] != old_manifest["source_sha256"]:
        raise RuntimeError("O arquivo bruto mudou entre os ciclos. Não compare conjuntos de dados distintos.")

    batches = [load_partition("batch_01")]
    if cycle == 2:
        batches.append(load_partition("batch_02"))
    train = pd.concat(batches, ignore_index=True)
    valid = load_partition("validation")
    X_train, y_train = train.drop(columns=[TARGET]), train[TARGET]
    X_valid, y_valid = valid.drop(columns=[TARGET]), valid[TARGET]

    selected, votes = vote_features(X_train, y_train)
    votes_path = REPORTS / f"votes_cycle_{cycle}.csv"
    votes.to_csv(votes_path, index=False)

    mlflow.set_tracking_uri(MLRUNS.resolve().as_uri())
    mlflow.set_experiment("WineQuality_N3")
    fitted = {}
    run_results = []
    for label, estimator in build_models().items():
        with mlflow.start_run(run_name=f"ciclo_{cycle}_{label}") as run:
            estimator.fit(X_train[selected], y_train)
            metrics = evaluate(estimator, X_valid[selected], y_valid)
            mlflow.log_params({
                "cycle": cycle,
                "algorithm": label,
                "train_rows": len(train),
                "feature_count": len(selected),
                "k_features_per_method": config["k_features_per_method"],
                "minimum_votes": config["minimum_votes"],
                "random_state": config["random_state"],
                "rf_n_estimators": config["rf_n_estimators"],
                "ridge_alpha": config["ridge_alpha"],
                "dataset_sha256": manifest["source_sha256"],
            })
            mlflow.log_metrics({
                "validation_mae": metrics["mae"],
                "validation_rmse": metrics["rmse"],
            })
            mlflow.set_tags({"dataset": "UCI winequality-red.csv", "evaluation_set": "fixed_validation"})
            mlflow.log_artifact(str(votes_path), artifact_path="feature_selection")
            mlflow.sklearn.log_model(estimator, artifact_path="model")
            fitted[label] = estimator
            run_results.append({
                "cycle": cycle,
                "model": label,
                "train_rows": len(train),
                "validation_mae": metrics["mae"],
                "validation_rmse": metrics["rmse"],
                "selected_features": ", ".join(selected),
                "run_id": run.info.run_id,
            })

    winner = min(run_results, key=lambda record: record["validation_mae"])
    incumbent_file = MODELS / "current.joblib"
    old_mae = None
    if incumbent_file.exists():
        incumbent = joblib.load(incumbent_file)  # Apenas arquivo confiável do projeto.
        old_mae = evaluate(
            incumbent["estimator"], X_valid[incumbent["features"]], y_valid
        )["mae"]
    promoted = old_mae is None or (
        winner["validation_mae"] < old_mae - config["min_mae_improvement"]
    )
    if promoted:
        joblib.dump({"estimator": fitted[winner["model"]], "features": selected}, incumbent_file)
        write_json(MODELS / "current.json", {
            "algorithm": winner["model"],
            "cycle": cycle,
            "validation_mae": winner["validation_mae"],
            "selected_features": selected,
            "features": selected,
            "mlflow_run_id": winner["run_id"],
            "dataset_sha256": manifest["source_sha256"],
            "promoted_at_utc": datetime.now(timezone.utc).isoformat(),
        })

    for record in run_results:
        record["winner_of_cycle"] = record["model"] == winner["model"]
        record["promoted"] = bool(promoted and record["winner_of_cycle"])
        record["incumbent_mae_before"] = old_mae
    history_file = REPORTS / "history.csv"
    new_history = pd.DataFrame(run_results)
    history = pd.concat([pd.read_csv(history_file), new_history], ignore_index=True) if history_file.exists() else new_history
    history.to_csv(history_file, index=False)

    summary = {
        "status": "completed",
        "cycle": cycle,
        "training_rows": len(train),
        "selected_features": selected,
        "winner": winner["model"],
        "winner_validation_mae": winner["validation_mae"],
        "incumbent_validation_mae_before": old_mae,
        "promoted": bool(promoted),
        "results": run_results,
    }
    if cycle == 2:
        # Uso único do teste para o modelo efetivamente vigente, depois da decisão.
        test = load_partition("test")
        package = joblib.load(incumbent_file)
        estimate = package["estimator"].predict(test[package["features"]])
        metrics = evaluate(package["estimator"], test[package["features"]], test[TARGET])
        pd.DataFrame({
            "actual_quality": test[TARGET].to_numpy(),
            "predicted_quality": estimate,
            "abs_error": abs(test[TARGET].to_numpy() - estimate),
            "residual": test[TARGET].to_numpy() - estimate,
        }).to_csv(REPORTS / "test_predictions.csv", index=False)
        active_meta = json.loads((MODELS / "current.json").read_text(encoding="utf-8"))
        write_json(REPORTS / "final_test.json", {
            "algorithm": active_meta["algorithm"],
            "model_promoted_in_cycle": active_meta["cycle"],
            "test_rows": len(test),
            **metrics,
        })
        summary["final_test_mae"] = metrics["mae"]

    write_json(REPORTS / f"summary_cycle_{cycle}.json", summary)
    write_json(state_path, {
        "last_cycle": cycle,
        "last_execution_utc": datetime.now(timezone.utc).isoformat(),
        "source_sha256": manifest["source_sha256"],
    })
    return summary

In [ ]:
resumo_ciclo_1 = run_cycle('1')
print('Ciclo 1:')
display(pd.DataFrame(resumo_ciclo_1.get('results', [])))
print('Decisão:', 'promovido' if resumo_ciclo_1.get('promoted') else resumo_ciclo_1['status'])

In [ ]:
resumo_ciclo_2 = run_cycle('2')
print('Ciclo 2:')
display(pd.DataFrame(resumo_ciclo_2.get('results', [])))
print('Decisão:', 'promovido' if resumo_ciclo_2.get('promoted') else 'mantido: não melhorou o MAE')
if 'final_test_mae' in resumo_ciclo_2:
    print(f"MAE final no conjunto de teste isolado: {resumo_ciclo_2['final_test_mae']:.4f}")

## 6. Histórico dos experimentos e interpretação

Os **4 runs** do MLflow contêm o algoritmo, o ciclo, o volume de treino, o hash do dataset, os atributos votados, as métricas da validação e o modelo. O arquivo `reports/history.csv` consolida as decisões e ajuda na defesa oral.

Para abrir a interface de rastreamento em um terminal na raiz do projeto:

```bash
mlflow ui --backend-store-uri ./mlruns
```

Depois acesse `http://127.0.0.1:5000`.

In [ ]:
historico = pd.read_csv(REPORTS / 'history.csv')
display(historico[['cycle', 'model', 'train_rows', 'validation_mae', 'validation_rmse', 'winner_of_cycle', 'promoted']].round(4))
assert len(historico) == 4 and set(historico['cycle']) == {1, 2}, 'Esperados 4 registros (2 modelos × 2 ciclos).'
meta_vigente = json.loads((MODELS / 'current.json').read_text(encoding='utf-8'))
print('Modelo vigente:', meta_vigente['algorithm'])
print('Ciclo em que foi promovido:', meta_vigente['cycle'])
print('Atributos vigentes:', meta_vigente['features'])
mlflow.set_tracking_uri(MLRUNS.resolve().as_uri())
runs_mlflow = mlflow.search_runs(experiment_names=['WineQuality_N3'])
print('Runs MLflow disponíveis:', len(runs_mlflow))

## 7. Dashboard no notebook — MAE, votação e resíduos

O painel reúne **comparação de MAE** entre modelos/ciclos, **votos por atributo**, **distribuição do erro absoluto** no teste e **nota real versus prevista**. Os valores são calculados no momento da execução e o gráfico é salvo em `evidencias/dashboard_notebook.png`.

**Leitura do erro:** por exemplo, MAE de 0,50 significa desvio absoluto médio de meio ponto na nota. Isso é um exemplo de interpretação da métrica, **não um resultado deste projeto**.

In [ ]:
historico = pd.read_csv(REPORTS / 'history.csv')
votos_1 = pd.read_csv(REPORTS / 'votes_cycle_1.csv')
votos_2 = pd.read_csv(REPORTS / 'votes_cycle_2.csv')
predicoes = pd.read_csv(REPORTS / 'test_predictions.csv')
fig, axes = plt.subplots(2, 2, figsize=(15, 10), constrained_layout=True)

# (1) Comparação direta dos MAEs por modelo e ciclo.
mae_grid = historico.pivot(index='cycle', columns='model', values='validation_mae').sort_index()
xx = np.arange(len(mae_grid.index))
width = 0.36
for pos, (model_name, vals) in enumerate(mae_grid.items()):
    bars = axes[0, 0].bar(xx + (pos - 0.5)*width, vals, width=width, label=model_name)
    axes[0, 0].bar_label(bars, fmt='%.3f', padding=3)
axes[0, 0].set_xticks(xx, [f'Ciclo {n}' for n in mae_grid.index])
axes[0, 0].set(title='MAE por modelo (validação fixa)', ylabel='Erro absoluto médio — menor é melhor')
axes[0, 0].legend()
axes[0, 0].set_ylim(0, max(mae_grid.max()) * 1.23)

# (2) Votação por atributo no último ciclo.
plot_votes = votos_2.set_index('feature')[['SelectKBest', 'RandomForest', 'RFE']]
heat = axes[0, 1].imshow(plot_votes.values, cmap='Purples', vmin=0, vmax=1, aspect='auto')
axes[0, 1].set_yticks(range(len(plot_votes.index)), plot_votes.index)
axes[0, 1].set_xticks(range(3), plot_votes.columns, rotation=20)
axes[0, 1].set_title('Seleção de atributos — votos do ciclo 2')
for i in range(len(plot_votes)):
    for j in range(3):
        axes[0, 1].text(j, i, str(int(plot_votes.iloc[i, j])), ha='center', va='center')

# (3) Erros absolutos no teste de fechamento.
axes[1, 0].hist(predicoes['abs_error'], bins=20, color='#88516b', edgecolor='white')
axes[1, 0].axvline(predicoes['abs_error'].mean(), color='black', linestyle='--', label='MAE final')
axes[1, 0].set(title='Distribuição dos erros absolutos (teste)', xlabel='|Nota real − prevista|', ylabel='Quantidade')
axes[1, 0].legend()

# (4) Real vs. estimado, somente depois do fechamento do segundo ciclo.
axes[1, 1].scatter(predicoes['actual_quality'], predicoes['predicted_quality'], alpha=0.55, color='#7b3550')
lo = min(predicoes['actual_quality'].min(), predicoes['predicted_quality'].min())
hi = max(predicoes['actual_quality'].max(), predicoes['predicted_quality'].max())
axes[1, 1].plot([lo, hi], [lo, hi], '--', color='black', label='Previsão perfeita')
axes[1, 1].set(title='Nota real × nota prevista (teste)', xlabel='Quality real', ylabel='Quality prevista')
axes[1, 1].legend()

fig.suptitle('Wine Quality N3 | Painel de Comparação e Erros', fontsize=16, fontweight='bold')
fig.savefig(EVIDENCIAS / 'dashboard_notebook.png', dpi=170, bbox_inches='tight')
plt.show()
print('Figura salva em:', EVIDENCIAS / 'dashboard_notebook.png')

## 8. Previsão de um vinho com o modelo vigente

O modelo salvo contém o **estimador treinado e a lista exata de atributos exigidos**. A função `predict_one` valida os campos e devolve uma nota contínua. No exemplo abaixo, usamos os **valores medianos do lote 1** para preencher a entrada. Edite o dicionário `medidas_vinho` para testar outra amostra.

In [ ]:
def predict_one(measurements: dict) -> float:
    path = MODELS / "current.joblib"
    if not path.exists():
        raise FileNotFoundError("Execute ao menos o ciclo 1 antes de prever.")
    package = joblib.load(path)  # Somente artefatos locais confiáveis.
    features = package["features"]
    missing = sorted(set(features) - set(measurements))
    if missing:
        raise ValueError(f"Medições ausentes: {missing}")
    data = pd.DataFrame([{feature: float(measurements[feature]) for feature in features}])
    return float(package["estimator"].predict(data)[0])

In [ ]:
vigente = json.loads((MODELS / 'current.json').read_text(encoding='utf-8'))
medianas_treino = pd.read_csv(PROCESSED / 'batch_01.csv').median(numeric_only=True)
medidas_vinho = {coluna: float(medianas_treino[coluna]) for coluna in vigente['features']}
display(pd.DataFrame([{'Atributo': atributo, 'Valor informado': valor} for atributo, valor in medidas_vinho.items()]))
print(f'Nota contínua prevista: {predict_one(medidas_vinho):.3f}')
print('A previsão é estatística e não substitui análise sensorial.')

## 9. Verificação final e arquivos de entrega

O checklist abaixo verifica artefatos locais após a execução. A **automação no GitHub Actions e o Streamlit são arquivos complementares presentes no pacote ZIP**, e precisam ser demonstrados executando-os. A existência do arquivo YAML não prova que o workflow já executou em um repositório remoto.

In [ ]:
checks = {
    '01. Origem/manifesto UCI': (REPORTS / 'dataset_manifest.json').exists(),
    '02. CSVs de treino/validacao/teste': all((PROCESSED / f'{x}.csv').exists() for x in ('batch_01','batch_02','validation','test')),
    '03. 3 seletores nos votos': all((pd.read_csv(REPORTS / f'votes_cycle_{i}.csv')[['SelectKBest','RandomForest','RFE']].sum() > 0).all() for i in (1,2)),
    '04. Voto 2/3 respeitado': all((pd.read_csv(REPORTS / f'votes_cycle_{i}.csv').query('selected == True')['votes'] >= 2).all() for i in (1,2)),
    '05. 2 modelos × 2 ciclos': len(pd.read_csv(REPORTS / 'history.csv')) == 4,
    '06. Histórico MLflow gravado': len(mlflow.search_runs(experiment_names=['WineQuality_N3'])) >= 4,
    '07. Workflow configurado no ZIP': (ROOT / '.github/workflows/pipeline_mlops.yml').exists(),
    '08. Modelo vigente e inferencia': (MODELS / 'current.joblib').exists() and (MODELS / 'current.json').exists(),
    '09. Teste isolado e relatorios': (REPORTS / 'final_test.json').exists() and (REPORTS / 'test_predictions.csv').exists(),
    '10. Dashboard extra': (EVIDENCIAS / 'dashboard_notebook.png').exists(),
}
display(pd.DataFrame([{'Critério': nome, 'Situação': 'OK' if sucesso else 'PENDENTE'} for nome, sucesso in checks.items()]))
print('Checklist:', sum(checks.values()), '/', len(checks))
print('Observação: workflow, dashboard externo e envio ao GitHub devem ser demonstrados fora deste notebook.')

## Referências, defesa e cuidados metodológicos

**Resumo para a apresentação:** problema de regressão da nota `quality`; ETL UCI e partição 60/20/20; votação independente de atributos 2/3 em dois ciclos; Ridge e Random Forest Regressor; comparação pelo MAE; rastreio dos quatro runs no MLflow; promoção estritamente condicionada à melhora; teste reservado; painel de comparação e inferência.

**Limitações:** não há novos vinhos em tempo real; a base reflete vinhos tintos desta coleta; correlações não provam causalidade; o modelo estima a nota do conjunto e não um parecer profissional. Toda conclusão numérica da apresentação deve corresponder aos arquivos produzidos pela execução real.

**Fontes:**
- [UCI Wine Quality — Cortez et al. (2009)](https://archive.ics.uci.edu/dataset/186/wine+quality)
- [Scikit-learn](https://scikit-learn.org/stable/)
- [MLflow Tracking](https://mlflow.org/docs/latest/tracking/)
- [Streamlit](https://docs.streamlit.io/)
- [GitHub Actions](https://docs.github.com/en/actions/reference/workflows-and-actions/workflow-syntax)

**Conteúdos do ZIP:** `src/*.py`, `config/params.json`, `dashboard.py`, `.github/workflows/pipeline_mlops.yml`, `tests/`, README e 5 notebooks complementares. Verifique também eventuais critérios adicionais do enunciado original da disciplina.